In [1]:
from datasets import load_dataset

dataset = load_dataset("Ashikan/diabetic-friendly-recipes")

unique_ners = set()

for sample in dataset['train']:
    ner_list = sample.get('NER', [])
    for ner in ner_list:
        unique_ners.add(ner)

sorted_ners = sorted(unique_ners)

c:\Users\jiaxi\anaconda3\envs\csye7380\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset["train"].to_json("diabetic_friendly_recipes.json", orient="records", lines=True)

dataset["train"].to_csv("diabetic_friendly_recipes.csv")


Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 14.74ba/s]


817265

In [3]:
import json
with open("ingredient_labels.json") as f:
    ingredient_labels = json.load(f)

In [4]:
from collections import defaultdict

# assuming ingredient_labels is a dictionary where keys are ingredient names and values are lists of labels
all_labels = set(label for labels in ingredient_labels.values() for label in labels)

In [5]:
import numpy as np

def count_labels(ingredients):
    label_count = defaultdict(int)

    if isinstance(ingredients, np.ndarray):
        ing_list = [i.strip().lower() for i in ingredients]
    elif isinstance(ingredients, list):
        ing_list = [str(i).strip().lower() for i in ingredients]
    elif isinstance(ingredients, str):
        ing_list = [i.strip().lower() for i in ingredients.split(",")]
    else:
        ing_list = []

    for ing in ing_list:
        for keyword, labels in ingredient_labels.items():
            if keyword in ing:
                for label in labels:
                    label_count[label] += 1

    return label_count


In [6]:
import pandas as pd

df = dataset["train"].to_pandas()

for label in all_labels:
    df[label] = 0

for idx, row in df.iterrows():
    counts = count_labels(row["ingredients"])
    for label, value in counts.items():
        df.at[idx, label] = value


In [7]:
condition = (
    (df["sweetener"] > 1) |
    (df["carbohydrate"] > 2) |
    (df["processed"] > 3) |
    (df["oil_fat"] > 2) |
    (df["meat"] > 1)
)

df["is_diabetic_friendly"] = (~condition).astype(int)

In [8]:
df.to_csv("diabetic_recipes_with_categories.csv", index=False)